### Importando pandas e carregando o arquivo CSV

In [8]:
import pandas as pd
import numpy as np

df_ec = pd.read_csv('ecommerce_customer_features.csv')
df_tg = pd.read_csv('ecommerce_customer_targets.csv')



### Unindo os DataFrame

In [11]:
df_uni = pd.merge(df_ec, df_tg, on='Customer_ID') #União dos 2 arquivos CSV
df_uni.head()

,Customer_ID,account_age_months,avg_order_value,total_orders,days_since_last_purchase,discount_usage_rate,return_rate,customer_support_tickets,loyalty_member,browsing_frequency_per_week,cart_abandonment_rate,product_review_score_avg,engagement_score,satisfaction_score,price_sensitivity_index,churned
0,0520df14-712d-4c69-a0c5-95a2e7dfc1ff,46,164.96,12,17,0.243,0.1720,0,No,6.1,0.430,5.00,6.58,9.43,3.7,No
1,a4013b3f-0688-4096-a194-6074be8ffec8,3,39.09,4,5,0.591,0.0808,1,No,4.1,0.183,4.44,6.25,8.50,6.9,No
2,eb870f2c-ed3d-4a21-a8ac-273fae69ea4f,29,37.42,8,47,0.212,0.1424,0,No,1.2,0.426,3.87,3.32,8.40,4.3,No
3,a7433451-8ea9-428a-9d80-679c6963b39f,35,62.64,9,3,0.699,0.0128,0,No,3.8,0.730,4.75,6.42,9.71,7.5,No
4,43f81935-49e3-44d3-94d1-5c4715738988,39,113.03,1,7,0.382,0.0232,0,No,5.4,0.613,5.00,6.48,9.92,5.0,No


### Filtrando os clientes críticos

In [7]:
df_critico = df_uni[(df_uni['churned'] == 'Yes') & (df_uni['days_since_last_purchase'] > 30) & (df_uni['total_orders'] < 3)]
len(df_critico) #Literalmente o que o nome diz, ele ve o comprimento da variável df_critico

448

### Média de compras


In [6]:
valor_medio = df_critico['avg_order_value'].mean()
print(valor_medio)

79.5520982142857


### Diagnóstico de Engajamento

In [14]:
df_media = df_uni.groupby(['loyalty_member', 'customer_support_tickets']).agg({
    'cart_abandonment_rate': 'mean', #Calcula a média dos produtos do carrinho de compras abandonados
    'engagement_score': 'mean', #Calcula a média dos engajamentos dos produtos
    'Customer_ID': 'count', #Conta quantos clientes tem no grupo
})

df_media

cart_abandonment_rate  \
loyalty_member customer_support_tickets                          
No             0                                      0.602604   
               1                                      0.604750   
               2                                      0.598986   
               3                                      0.588141   
               4                                      0.601268   
               5                                      0.551867   
               6                                      0.700000   
Yes            0                                      0.597546   
               1                                      0.603381   
               2                                      0.591828   
               3                                      0.647365   
               4                                      0.685187   
               5                                      0.958000   

                                         engagement_score  Customer_ID  
loyalty_member customer_support_tickets                                 
No             0                                 4.843321         2171  
               1                                 4.811753         1706  
               2                                 4.820130          693  
               3                                 4.822773          256  
               4                                 4.690000           71  
               5                                 4.818000           15  
               6                                 4.983333            3  
Yes            0                                 5.178753          489  
               1                                 5.121000          370  
               2                                 5.294076          157  
               3                                 4.782115           52  
               4                                 4.366250           16  
               5                                 6.350000            1

### Perfis de Clientes

In [21]:
condicoes = [
    (df_uni['account_age_months'] < 12) & (df_uni['browsing_frequency_per_week'] > 4), # Procura pessoas que tem uma conta no site a menos de um ano e acessam mais de 4 vezes por semana
    (df_uni['account_age_months'] >= 24) & (df_uni['browsing_frequency_per_week'] <= 2) # Procura pessoas que tem contas de mais de 2 anos e entram menos que 2 vezes na semana
]

contas = ['Clientes_Novos', 'Contas_Inativas']

df_uni['Perfil_do_Cliente'] = np.select(condicoes, contas, default='Clientes_Rotativos') # Criação de uma nova coluna. O np.select testa todas as linhas de uma vez, coloca a resposta certa e cria a coluna nova padrão

In [22]:
df_uni['Perfil_do_Cliente'].value_counts()

Perfil_do_Cliente
Clientes_Rotativos    4495
Contas_Inativas       1183
Clientes_Novos         322
Name: count, dtype: int64